# Project 2 – Traffic and Air Quality in New York City  
### Exploring whether busier roads mean dirtier air

In this project, I use two datasets from NYC Open Data to explore a simple but meaningful question:  
**How does traffic volume relate to air quality levels across New York City?**

Air pollution—especially fine particulate matter (PM2.5)—is a significant urban health concern. At the same time, New York City streets carry high daily traffic volumes, and vehicles are a major source of emissions. It feels intuitive that more cars might lead to worse air quality, but does the data actually support that assumption? This project investigates that relationship using publicly available datasets.

### **Research Question**  
**How does traffic volume relate to air quality levels across New York City?**

### **Hypothesis**  
Higher traffic volume is associated with worse air quality.

### **Data Overview**  
To answer this question, I work with two datasets:

1. **NYC Air Quality Data**  
   Contains annual and seasonal measurements of key pollutants (including PM2.5, NO₂, and ozone) across NYC neighborhoods.  
   In this project, I focus on **annual PM2.5**, which provides a reliable indicator of long-term changes in air quality.

2. **NYC Traffic Volume Counts**  
   Contains hourly vehicle counts for specific roadway segments across the city.  
   From this dataset, I compute **annual average daily traffic** to align with the time scale of the air quality data.

Although these datasets describe different aspects of urban life—pollution and mobility—they both contain a time dimension that allows them to be merged. By converting both datasets to annual measures (annual PM2.5 and annual average daily traffic), I can visualize and compare long-term trends in NYC air quality and traffic activity.

### Data Sources

Both datasets used in this project come from **NYC Open Data**, the city’s open data portal.

- **Air Quality Data (PM2.5, NO2, Ozone, etc.)**  
  Source: NYC Environment & Health Data Portal  
  https://data.cityofnewyork.us/Environment/Air-Quality/c3uy-2p5r

- **NYC Traffic Volume Counts (Hourly Traffic Data)**  
  Source: NYC Department of Transportation (NYC DOT)  
  https://data.cityofnewyork.us/Transportation/Traffic-Volume-Counts/btm5-ppia/about_data

I downloaded both CSV files directly from NYC Open Data and work with the cleaned versions locally in this notebook.

## Dataset A: Air Quality

The air quality dataset comes from NYC Open Data and includes measurements of several key pollutants—such as PM2.5, nitrogen dioxide (NO₂), and ozone (O₃)—across different geographic areas in New York City. Each row corresponds to a pollutant measured in a specific location during a defined time period (e.g., “Annual Average 2020” or “Summer 2023”), and contains:

- the pollutant name  
- the measurement unit  
- the geographic area (UHF or community district)  
- the time period of the measurement  
- the numerical pollutant value  

For this project, I focus on **fine particulate matter (PM2.5)**. PM2.5 is a widely used indicator of air pollution because the particles are small enough to enter the bloodstream and are strongly linked to asthma, cardiovascular disease, and premature mortality.

Although the dataset provides both seasonal and annual values, I use the **annual PM2.5 averages** for this analysis. Annual data provide a consistent year-to-year measure of air quality and align with the annual traffic metrics derived from the traffic dataset. This produces a clean, comparable time series that allows us to investigate long-term trends in PM2.5 and how they relate to citywide traffic activity.

#### Loading and Previewing the Dataset

In [2]:
import pandas as pd

# Load Dataset A: Air Quality
aq = pd.read_csv("Air_Quality_20251121.csv")

# Preview the data
aq.head()

,Unique ID,Indicator ID,Name,Measure,Measure Info,Geo Type Name,Geo Join ID,Geo Place Name,Time Period,Start_Date,Data Value,Message
0,878218,386,Ozone (O3),Mean,ppb,UHF42,402,West Queens,Summer 2023,06/01/2023,34.365989,NaN
1,876975,375,Nitrogen dioxide (NO2),Mean,ppb,UHF42,501,Port Richmond,Summer 2023,06/01/2023,11.331992,NaN
2,876900,375,Nitrogen dioxide (NO2),Mean,ppb,UHF42,207,East Flatbush - Flatbush,Summer 2023,06/01/2023,12.020333,NaN
3,877140,375,Nitrogen dioxide (NO2),Mean,ppb,CD,205,Fordham and University Heights (CD5),Summer 2023,06/01/2023,14.123178,NaN
4,874556,365,Fine particles (PM 2.5),Mean,mcg/m3,UHF34,410,Rockaways,Summer 2023,06/01/2023,8.150637,NaN


The preview shows the structure of the dataset, including:
- the pollutant name (Name),
- the measurement unit (Measure Info),
- the geographic identifier (Geo Join ID and Geo Place Name),
- the time period of the measurement (Time Period),
- and the numerical pollutant value (Data Value).

Since this project focuses on long-term changes in air quality, I use the annual PM2.5 measurements from this dataset and aggregate them into a yearly time series for comparison with annual traffic trends.

### Cleaning and Preparing the Air Quality Data

To prepare the air quality dataset for analysis, I first filter the data to keep only measurements of **fine particulate matter (PM2.5)**. PM2.5 is the pollutant most directly relevant to air quality and health outcomes, and it provides a reliable indicator for examining long-term pollution trends.

Although the dataset contains both seasonal (e.g., “Summer 2023”) and annual (e.g., “Annual Average 2020”) measurements, this project focuses on **annual PM2.5 values** to align with the annual traffic metrics computed from the traffic dataset. Using annual measurements provides a consistent, year-to-year basis for comparison.

The code below filters the dataset to PM2.5 and computes the **citywide annual average PM2.5** for each year.

In [3]:
# Filter for PM2.5 pollutant only
aq_pm25 = aq[aq["Name"] == "Fine particles (PM 2.5)"].copy()

# Compute average PM2.5 for each time period (annual or seasonal)
pm25_season = aq_pm25.groupby("Time Period")["Data Value"].mean().reset_index()
pm25_season.rename(columns={"Data Value": "pm25"}, inplace=True)

pm25_season

,Time Period,pm25
0,Annual Average 2009,10.977801
1,Annual Average 2010,10.069574
2,Annual Average 2011,10.585390
3,Annual Average 2012,9.440071
4,Annual Average 2013,9.135248
5,Annual Average 2014,9.402199
6,Annual Average 2015,9.038726
7,Annual Average 2016,7.882979
8,Annual Average 2017,7.737801
9,Annual Average 2018,7.384326


This produces a table of annual PM2.5 averages across New York City—an essential input for merging with the annual traffic volume dataset in the next step.

### Extracting Season and Year from the Air Quality Time Period

The air quality dataset encodes time information in the `Time Period` column using mixed formats, such as:

- **Annual Average 2015**
- **Summer 2019**
- **Winter 2018–19**

To standardize these entries and create a consistent temporal index, I extract two pieces of information:

1. **Season** — whether the observation corresponds to Annual, Summer, or Winter.  
2. **Year** — the year associated with the measurement.  
   - For annual values, the year appears directly (e.g., “Annual Average 2015”).  
   - For seasonal values like “Summer 2017,” the year is also explicit.  
   - For winter periods that span two years (e.g., “Winter 2018–19”), I assign the **second year** (“2019”), which reflects the convention used in the dataset.

The following code applies two helper functions to parse the season and the appropriate year from each `Time Period` string.

In [4]:
import re

# Make a copy to avoid modifying the raw dataset
pm25 = pm25_season.copy()


# Extract Season
def get_season(tp):
    if "Summer" in tp:
        return "Summer"
    elif "Winter" in tp:
        return "Winter"
    elif "Annual" in tp:
        return "Annual"
    else:
        return None


pm25["season"] = pm25["Time Period"].apply(get_season)


# Extract Year
def extract_year(tp):
    # Annual Average 2009
    if "Annual" in tp:
        return int(tp.split()[-1])
    # Summer 2010
    if "Summer" in tp:
        return int(tp.split()[-1])
    # Winter 2008-09 → we pick the second year (= 2009)
    if "Winter" in tp:
        yr = tp.split()[-1]
        start, end = yr.split("-")
        # winter 2008-09 => take "09" and map to 2009
        end_year = int("20" + end) if len(end) == 2 else int(end)
        return end_year
    return None


pm25["year"] = pm25["Time Period"].apply(extract_year)

pm25.head()

,Time Period,pm25,season,year
0,Annual Average 2009,10.977801,Annual,2009
1,Annual Average 2010,10.069574,Annual,2010
2,Annual Average 2011,10.585390,Annual,2011
3,Annual Average 2012,9.440071,Annual,2012
4,Annual Average 2013,9.135248,Annual,2013


This produces a clean table with three key fields—**PM2.5 value**, **season**, and **year**—allowing the annual PM2.5 measurements to be aligned with the yearly traffic dataset in the next section.

## Dataset B: Traffic Volume Counts

The second dataset comes from NYC Open Data, published by the NYC Department of Transportation (NYC DOT). It contains hourly vehicle counts collected on specific roadway segments across New York City. Each row represents one traffic count session for a given street segment on a specific date.

This dataset provides detailed insight into road usage, congestion, and daily mobility patterns, which makes it a strong complement to the air quality dataset.

The dataset includes the following information:

- roadway name and cross streets,
- the direction of traffic flow,
- the date of the observation,
- and **24 hourly traffic count columns** (e.g., “12:00–1:00 AM”, “1:00–2:00 AM”, ..., “11:00–12:00 PM”).

Taken together, these hourly counts provide a quantitative picture of how busy each roadway segment was during each 24-hour period.  

Because my research question examines whether busier roads are associated with worse air quality, I aggregate the hourly data into a single **daily traffic volume** per row, and later compute the **annual average traffic volume** to align with the annual PM2.5 values.



#### Loading and Previewing the Dataset

In [5]:
# Load Dataset B: Traffic Volume Counts
traffic = pd.read_csv("Traffic_Volume_Counts_20251121.csv")

# Preview
traffic.head()

,ID,SegmentID,Roadway Name,From,To,Direction,Date,12:00-1:00 AM,1:00-2:00AM,2:00-3:00AM,...,2:00-3:00PM,3:00-4:00PM,4:00-5:00PM,5:00-6:00PM,6:00-7:00PM,7:00-8:00PM,8:00-9:00PM,9:00-10:00PM,10:00-11:00PM,11:00-12:00AM
0,1,15540,BEACH STREET,UNION PLACE,VAN DUZER STREET,NB,01/09/2012,20,10,11,...,104,105,147,120,91,83,74,49,42,42
1,2,15540,BEACH STREET,UNION PLACE,VAN DUZER STREET,NB,01/10/2012,21,16,8,...,102,98,133,131,95,73,70,63,42,35
2,3,15540,BEACH STREET,UNION PLACE,VAN DUZER STREET,NB,01/11/2012,27,14,6,...,115,115,130,143,106,89,68,64,56,43
3,4,15540,BEACH STREET,UNION PLACE,VAN DUZER STREET,NB,01/12/2012,22,7,7,...,71,127,122,144,122,76,64,58,64,43
4,5,15540,BEACH STREET,UNION PLACE,VAN DUZER STREET,NB,01/13/2012,31,17,7,...,113,126,133,135,102,106,58,58,55,54


The preview confirms that the dataset contains the hourly traffic columns, the observation date, and the necessary metadata for constructing yearly traffic trends.

### Cleaning and Preparing the Traffic Data

The traffic dataset records 24 hourly vehicle counts for each roadway segment on a given date.  
To align this dataset with the annual PM2.5 data, I transform the raw hourly counts into a clean **annual traffic time series**.

This preprocessing involves several steps:

1. **Convert the `Date` column to a datetime format**  
   Ensures consistent handling of time information.

2. **Identify all hourly traffic columns**  
   These columns contain “:” in their names (e.g., “12:00–1:00 AM”), making them easy to detect programmatically.

3. **Convert hourly counts to numeric values**  
   Some columns are stored as strings; converting them prevents errors during aggregation.

4. **Compute a daily total traffic volume**  
   By summing across all 24 hourly columns.

5. **Extract the observation year**  
   Used later to compute annual averages.

6. **Aggregate traffic volume at the yearly level**  
   Produces a single annual average traffic value for each year in the dataset.


In [6]:
# 1. Convert Date to datetime
traffic["Date"] = pd.to_datetime(traffic["Date"], errors="coerce")

# 2. Identify hourly columns (columns with ":")
hour_cols = [col for col in traffic.columns if ":" in col]

# 3. Convert hourly columns to numeric, coercing errors to NaN
traffic[hour_cols] = traffic[hour_cols].apply(pd.to_numeric, errors="coerce")

# 4. Compute daily traffic volume
traffic["daily_volume"] = traffic[hour_cols].sum(axis=1)

# 5. Extract year
traffic["year"] = traffic["Date"].dt.year

# 6. Aggregate by year
traffic_yearly = traffic.groupby("year")["daily_volume"].mean().reset_index()
traffic_yearly.rename(columns={"daily_volume": "avg_daily_traffic"}, inplace=True)

traffic_yearly.head()

,year,avg_daily_traffic
0,2012,5835.479009
1,2013,5121.041958
2,2014,5928.831625
3,2015,5834.123797
4,2016,5509.704639


This results in a clean annual traffic dataset, which can be directly merged with the annual PM2.5 time series to analyze long-term relationships between traffic volume and air quality in New York City.

### Selecting Annual PM2.5 Values

After extracting both the season and the year from each `Time Period` entry, I isolate the **annual PM2.5 measurements**. Seasonal PM2.5 values (Summer/Winter) are not used in this analysis, since the traffic dataset is aggregated at the yearly level. This step produces a clean two-column dataset containing one PM2.5 value per year.

In [7]:
pm25_annual = pm25[pm25["season"] == "Annual"][["year", "pm25"]]
pm25_annual.head()

,year,pm25
0,2009,10.977801
1,2010,10.069574
2,2011,10.585390
3,2012,9.440071
4,2013,9.135248


## Merging the Annual Air Quality and Traffic Datasets

After preparing both datasets, I now merge the **annual PM2.5 data** with the **annual average daily traffic volumes**.  
Both datasets use the same temporal index—**year**—which makes this step straightforward.

Merging the two sources creates a unified table containing:

- the year,
- the average PM2.5 concentration for that year,
- the average daily traffic volume for that year.

This consolidated dataset allows me to directly examine whether trends in traffic activity correspond with trends in air pollution over time.

In [8]:
merged = pd.merge(pm25_annual, traffic_yearly, on="year", how="inner")
merged

,year,pm25,avg_daily_traffic
0,2012,9.440071,5835.479009
1,2013,9.135248,5121.041958
2,2014,9.402199,5928.831625
3,2015,9.038726,5834.123797
4,2016,7.882979,5509.704639
5,2017,7.737801,6407.110014
6,2018,7.384326,5601.264313
7,2019,7.012553,6449.370142
8,2020,6.321702,5855.166932
9,2021,6.761250,4725.341564


The merged table below shows that both datasets span 2012–2021, yielding a complete annual time series for analysis and visualization.

## Visualizing the Relationship Between Traffic Volume and PM2.5

With the air quality and traffic datasets merged on a shared annual timeline, I now visualize how both variables evolve over time. Because traffic volume and PM2.5 concentration are measured on different scales, I use a **dual-axis plot**:

- **Left y-axis** shows annual average daily traffic volume (bar chart),
- **Right y-axis** shows annual average PM2.5 concentration (line chart).

This visualization allows the two trends to be compared directly without rescaling either variable.

In [13]:
import plotly.graph_objects as go
from IPython.display import HTML

In [14]:
import plotly.graph_objects as go

fig = go.Figure()

# Traffic volume bar chart
fig.add_trace(
    go.Bar(
        x=merged["year"],
        y=merged["avg_daily_traffic"],
        name="Avg Daily Traffic",
        marker_color="steelblue",
        yaxis="y1",
    )
)

# PM2.5 line chart
fig.add_trace(
    go.Scatter(
        x=merged["year"],
        y=merged["pm25"],
        name="PM2.5 (µg/m³)",
        mode="lines+markers",
        marker_color="firebrick",
        yaxis="y2",
    )
)

# Layout
fig.update_layout(
    title="Annual Traffic Volume vs PM2.5 Levels in New York City",
    xaxis_title="Year",
    yaxis=dict(title="Average Daily Traffic", side="left"),
    yaxis2=dict(title="PM2.5 (µg/m³)", overlaying="y", side="right"),
    height=500,
    legend=dict(title="Measures"),
)


HTML(fig.to_html(include_plotlyjs="cdn", full_html=False))

### Interpretation

The figure shows two distinct long-term patterns in New York City from 2012 to 2021:

- **PM2.5 levels decline consistently over the decade.**  
  Concentrations fall from above 9 µg/m³ in 2012 to around 6–7 µg/m³ by 2021, reflecting a steady improvement in air quality.

- **Traffic volumes fluctuate without a clear downward trend.**  
  Average daily traffic remains relatively stable across most years, with temporary drops in 2013 and 2020. The sharp decline in 2020 likely reflects COVID-19 mobility restrictions rather than long-term structural change.

- **The two trends do not move together.**  
  Years with higher traffic (e.g., 2014, 2018, 2019) do not correspond to higher PM2.5, and years with declining PM2.5 do not show parallel declines in traffic.

**Overall, the data suggests that traffic volume alone does not explain changes in PM2.5 levels in NYC during this period.** This implies that broader emissions regulations, cleaner vehicle technologies, and reductions from non-traffic pollution sources likely played a more significant role in driving air quality improvements.

## Conclusion

This project explored whether higher traffic volumes are associated with worse air quality in New York City. Using annual PM2.5 concentrations and annual average daily traffic volumes from NYC Open Data, I compared trends from 2012 to 2021.

The results show that PM2.5 levels declined steadily over the decade, while traffic volumes fluctuated without a matching downward trend. Years with higher traffic did not consistently correspond to higher pollution levels. This suggests that improvements in air quality were likely driven by broader factors—such as cleaner vehicle technologies, stricter emissions standards, and reductions from non-traffic pollution sources—rather than reductions in traffic volume alone.

Overall, the data indicates that **traffic volume is not the primary driver of year-to-year changes in PM2.5 levels**, and that urban air quality improvements depend on a combination of regulatory, technological, and environmental factors.